# 07 Choosing Statevector, MPS, Or Tensor Network

Goal: compare the summaries produced by local statevector, MPS, and tensor-network execution on the same small circuit.

Use this tutorial before choosing a runtime for a larger example.

## Decision guide

- Use statevector when the circuit is small enough and you want exact dense simulation.
- Use MPS when the workload has one-dimensional or low-entanglement structure.
- Use tensor networks when contraction structure and slicing are more important than storing a full state.

This notebook is local-only. It does not make distributed scalability claims.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "flagquantum").exists():
        sys.path.insert(0, str(candidate))
        break

import torch
import flagquantum as fq
import flagquantum.simulation.mps as fqmps
import flagquantum.simulation.tensor_network as fqtn

## Build one circuit

The same `fq.Circuit` object can feed multiple local runtime paths.

In [ ]:
theta = torch.tensor([0.2, -0.1, 0.3, 0.4])

def build_circuit() -> fq.Circuit:
    circuit = fq.Circuit(4)
    circuit.ry(0, theta=theta[0])
    circuit.ry(1, theta=theta[1])
    circuit.cx(0, 1)
    circuit.cx(1, 2)
    circuit.rzz(2, 3, theta=theta[2])
    circuit.rx(3, theta=theta[3])
    return circuit

circuit = build_circuit()
print(circuit)

## Statevector summary

Statevector stores the full dense state. For small circuits this is the easiest exact reference.

In [ ]:
statevector_summary = circuit.plan().summary()
statevector_summary

## MPS summary

MPS exposes bond dimensions, truncation, and canonicalization diagnostics. Those numbers matter more than a single runtime number.

In [ ]:
mps_state = fqmps.run_mps(circuit, max_bond=8)
mps_state.summary()

## Tensor-network summary

Tensor-network execution exposes graph size, contraction path length, cost, and peak size estimates.

In [ ]:
tn_state = fqtn.run_tensor_network(circuit)
tn_state.summary()

## What to report

When you publish an example, report the runtime mode, device, dtype, memory or structural summary, and whether the execution was local, sliced, replicated, or sharded.